# Week 12 Live Coding: Where Should the Last \$50 Million Go?

Seven states, 93 electoral votes, and you need 44 of them. Today we:

1. Simulate 200,000 elections, one step at a time, and count how often you reach 270.
2. See where the analytics director's 0.07 came from.
3. Build diminishing returns, and run the first round of the algorithm by hand.
4. Run all fifty rounds and see where the \$50 million goes.

### Before you start

**There is no problem set this week.** Before you leave, print this notebook to PDF and upload it to the **in-class** assignment on Canvas (the steps are at the bottom). That is your attendance credit. It is not graded on being right.

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Two datasets, described in the comments below. We start with `states`; `campaigns` comes back in Part 2.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# states: the seven swing states, one row each. Its columns are the
# state's electoral votes and your margin in the polling average
# (+ means you lead, - means you trail).
states = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/swing_states.csv')

# campaigns: the analytics director's data, 60 past campaigns, one
# row each. spend_advantage_m is how many $M more the campaign spent
# on ads in a state than its opponent; margin_shift_pp is how many
# points its margin there moved. We use it in Part 2.
campaigns = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk12_presidential_allocation/data/ad_spending_effects.csv')

states

You already hold 226 electoral votes. The simulation needs each state's electoral votes and margin as plain lists of seven numbers, in the same order as the table: PA, GA, NC, MI, AZ, WI, NV.

In [ ]:
held = 226                                  # electoral votes you hold

# .values: just the seven numbers, without the table around them
state_evs = states['electoral_votes'].values
margin = states['margin'].values

print('electoral votes:', state_evs)
print('your margin:', margin)

## Part 1: Simulate the election, one step at a time

Polls miss in two ways at once. A **national** miss hits every state in the same direction (in 2020 the polls overstated Biden almost everywhere). A **state** miss is different in each state.

We simulate 200,000 elections. Each step below is one step from the slides, and each cell prints the first five elections so you can see what the step did.

**Step 1: draw a national miss for each election.**

In [ ]:
np.random.seed(2026)        # same seed as earlier weeks
n_elections = 200000

# np.random.normal(average, SD, size) draws random numbers. The size
# (n_elections, 1) means a grid with 200,000 rows and 1 column:
#   one row per election,
#   one column, because there is only one national miss per election.
national_miss = np.random.normal(0, 3.5, (n_elections, 1))

# show the first five elections as a table with one column
pd.DataFrame(national_miss, columns=['national']).round(1).head()

One number per election. Election 1's polls are off by that much **in every state**.

**Step 2: draw a separate miss for each state.**

In [ ]:
# Now the size is (n_elections, 7): 200,000 rows and 7 columns.
#   one row per election,
#   one column per state, because each state gets its own miss.
state_miss = np.random.normal(0, 2.0, (n_elections, 7))

# show the first five elections as a table, one column per state
pd.DataFrame(state_miss, columns=states['state']).round(1).head()

Seven numbers per election, all different.

**Step 3: add both misses to your poll margin.**

In [ ]:
# Python lines the three pieces up for you:
#   margin has 7 numbers, one per state: each goes into its own column.
#   national_miss has 1 column: it is added to all 7 columns of its row.
#   state_miss has 7 columns: each state gets its own.
result = margin + national_miss + state_miss

pd.DataFrame(result, columns=states['state']).round(1).head()

Your result in each state, in each election. Pick election 1 and check one state by hand: poll margin + national miss + that state's miss.

**Step 4a: did you win each state?**

In [ ]:
# True where your result is above 0: you carried that state
won = result > 0

pd.DataFrame(won, columns=states['state']).head()

**Step 4b: add up your electoral votes in each election.**

In [ ]:
# won * state_evs: True counts as 1 and False as 0, so a state you
# won counts its electoral votes and a state you lost counts 0.
# .sum(axis=1): add across each row, which is within one election.
your_evs = held + (won * state_evs).sum(axis=1)

your_evs[:5]                # your electoral votes, first five elections

Check one against the True/False table above: 226 plus the electoral votes of every state marked True.

**Now count across all 200,000 elections.**

In [ ]:
# your_evs >= 270 is True or False for each election. The mean of a
# True/False list is the share that are True: your chance of 270.
print('average electoral votes:', round(your_evs.mean(), 1))
print('chance of reaching 270:',
      round(100 * (your_evs >= 270).mean(), 1), '%')

*Check: 261.8 and 39.5%.*

These are random draws, so a few numbers in this notebook differ slightly from the slides. The conclusions do not.

You never get 261.8 electoral votes. Here is what you do get.

In [ ]:
# a histogram: how many elections ended at each electoral-vote total
plt.hist(your_evs, bins=50, color='gray')
plt.axvline(270, color='#B58900')   # the 270 line
plt.xlabel('Your electoral votes')
plt.ylabel('Number of simulated elections')
plt.show()

The two tall bars are losing all seven states (226) and winning all seven (319). That is the national miss: when the polls are wrong, they are wrong in every state at once.

## Part 2: Where the analytics director's 0.07 came from

Sixty past campaigns, one row each: how many million dollars more the campaign spent on ads in a state than its opponent, and how many points its margin in that state moved.

In [ ]:
campaigns.head(10)

In [ ]:
# a regression: how much the margin moved for each extra $1M spent
reg = smf.ols('margin_shift_pp ~ spend_advantage_m',
              data=campaigns).fit()
coef = reg.params['spend_advantage_m']   # points of margin per $1M

print('$1M moves a state', round(coef, 3), 'points')

*Check: 0.071.*

**Nobody randomized any of this.** Campaigns choose where to spend. If they put more money where they were already gaining, this line credits the ads for gains they did not cause.

We build the plan on 0.07 anyway. The slides ask what happens if it is wrong.

## Part 3: Diminishing returns, and the first million by hand

Each extra million should buy a little less than the one before. The simplest way to say that: the margin you buy is 0.07 times the **square root** of the millions you spend. (The slides use a different curve with the same idea.)

In [ ]:
# np.sqrt is the square root. The square root grows more and more
# slowly: sqrt(1) = 1, sqrt(4) = 2, sqrt(100) = 10. So $100M buys
# only 10 times what $1M buys, not 100 times.
for d in [1, 10, 25, 50]:
    bought = 0.07 * np.sqrt(d)
    print('$' + str(d) + 'M buys', round(bought, 2), 'points.',
          'A straight line says', round(0.07 * d, 2))

*Check: \$50M buys 0.49 points, against 3.5 on a straight line.*

Now round 1 of the algorithm. For each state: put \$1M there and nowhere else, rerun the same 200,000 elections, and see how much your chance of reaching 270 goes up.

In [ ]:
before = (your_evs >= 270).mean()     # 39.5%, nothing spent yet
gains = []                            # one number per state

for i in range(7):                    # i = 0 is PA, 1 is GA, ... 6 is NV
    spend = np.zeros(7)               # $M in each state: seven zeros
    spend[i] = 1                      # put $1M in state i only

    # each state's new margin: its poll margin plus what its spending
    # buys, 0.07 x the square root of the $M spent there. spend holds
    # seven numbers, so this gives seven new margins at once. A state
    # with $0 gets sqrt(0) = 0, so its margin does not change.
    new_margin = margin + coef * np.sqrt(spend)

    # steps 3 and 4 again, on the same 200,000 elections
    won_new = (new_margin + national_miss + state_miss) > 0
    evs_new = held + (won_new * state_evs).sum(axis=1)

    # how much your chance of reaching 270 went up, in points
    gain = 100 * ((evs_new >= 270).mean() - before)
    gains.append(round(gain, 3))

states['first $1M'] = gains           # add the gains as a new column
states

*Check: Pennsylvania +0.259, Nevada +0.035.*

Pennsylvania comes first: big and close. Nevada is closer than Georgia, Arizona and North Carolina and comes last, because six electoral votes rarely decide the election.

## Part 4: All fifty rounds

The algorithm repeats round 1 fifty times. Each round it tries the next \$1M in each of the seven states, and gives it to the state where it raises your chance of reaching 270 the most. This takes a few seconds.

In [ ]:
plan = np.zeros(7)              # $M given to each state so far: none

for r in range(50):             # 50 rounds, one $1M each
    scores = []                 # this round: one score per state

    for i in range(7):          # try this round's $1M in each state
        # spending if the $1M went to state i: everything given out
        # so far (plan), plus $1M more in state i (extra)
        extra = np.zeros(7)
        extra[i] = 1
        spend = plan + extra

        # same three lines as round 1: new margins, rerun the same
        # 200,000 elections, and score the try by your chance of 270
        new_margin = margin + coef * np.sqrt(spend)
        won_new = (new_margin + national_miss + state_miss) > 0
        evs_new = held + (won_new * state_evs).sum(axis=1)
        scores.append((evs_new >= 270).mean())

    # scores now holds seven chances of 270, one per state.
    # np.argmax gives the position of the biggest one: the state where
    # this $1M helped most. That state gets the million, and the next
    # round starts from the new plan.
    best = np.argmax(scores)
    plan[best] = plan[best] + 1

states['plan'] = plan.astype(int)   # astype(int): whole millions
states

*Check: PA 30, GA 7, MI 5, WI 4, NC 2, AZ 1, NV 1.*

How much did the \$50 million buy?

In [ ]:
# the plan's margins, then the same 200,000 elections one more time
new_margin = margin + coef * np.sqrt(plan)
won_new = (new_margin + national_miss + state_miss) > 0
evs_new = held + (won_new * state_evs).sum(axis=1)

print('chance of 270, nothing spent:', round(100 * before, 1), '%')
print('chance of 270, with the plan:',
      round(100 * (evs_new >= 270).mean(), 1), '%')

*Check: 39.5% and 41.9%.*

Pennsylvania gets the most by far: it is big and close. But the square root makes the first million in any state buy the most, so some money goes to every state. The slides use a gentler curve and the money goes to only three states (Pennsylvania, Michigan and Wisconsin, exactly the 44 you need). **The shape you assume for diminishing returns changes the plan.**

---

## Before you leave

1. **Runtime → Restart session and run all.** It runs the notebook from top to bottom, in order. The Part 4 cell takes a few seconds.
2. **Check that every cell ran.** Scroll from the top. Every code cell should show its output below it.
3. **File → Print → Save as PDF.**
4. **Upload the PDF** to the **in-class** assignment on Canvas.